### Type author seats (oxjob #2437)

The name rules' half of the nightly agent typing. Proposes `software-agent` seats and builds a card for each (`openalex.authors.ai_byline_cards`). The next End 2 End task (`notebooks/authors/confirm_author_types.py`) has a model confirm them and writes `openalex.works.work_author_types`'s software-agent rows: proposed seats the model confirms. Runs after Author_Affiliations (UpdateWorkAuthors brings `work_authors` to tonight's bylines); Authorships waits for the confirmer through a gate and runs on last night's types when either step fails.

1. `openalex.authors.type_ai_byline_name`: the name rules, a Python UDF **generated** from oxjobs #2437 `work/build/ai_names.py` (tests: `test_ai_names.py`) by `work/build/patch_walden_typer.py`. Edit the module, then regenerate; `patch_walden_typer.py <walden> --check` fails when they differ.
2. Candidates: seats of `work_authors` whose lowered name passes the module's SQL prefilter (keeps the UDF off ~1.5B seats), with the name `openalex_works_base` prints at that seat (rows for seats a byline has since lost drop out), the work's title and its dates.
3. One verdict per distinct name: the only statement that calls the UDF (a warehouse statement may call at most 5).
4. Proposed seats: software-agent verdicts, except a seat whose name is the work's title, or a work whose date is more than 31 days before the product named launched. The date is the publication date; the date OpenAlex first saw the work stands in only when the publication date is missing or impossible.
5. Cards: the bar's card fields for every proposed seat (byline, title, venue, publisher, year, work type, DOI, the seat's affiliation text). The confirmer renders and hashes them, and asks only about cards it has no answer for.

Organization rows come from another job (`TypeOrganizationSeats`); this step never touches them.
Revert: take Type_Author_Seats, Confirm_Author_Types and Author_Types_Gate out of `jobs/walden_end2end.yaml`; the build keeps reading `openalex.works.work_author_types` as last written.

In [ ]:
-- oxjob #2437: GENERATED from oxjobs #2437 work/build/ai_names.py (ai-names-v4) by work/build/patch_walden_typer.py.
-- Edit the module and its tests, then regenerate; don't edit this cell by hand.
CREATE OR REPLACE FUNCTION openalex.authors.type_ai_byline_name(s STRING)
RETURNS STRUCT<type: STRING, rule: STRING, products: ARRAY<STRING>, launch: STRING>
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #2437: types a printed byline name software-agent (names an AI product) or organization (an AI maker alone), else null. Source + tests: oxjobs #2437 work/build/ai_names.py'
AS $$
"""AI-name list and the seat typer for author type (oxjob #2437, 21 Nov 2026).

Source of truth for which byline seats OpenAlex types `software-agent` from the printed name alone. The walden UDF is
generated from this file (build_udf.py); tests live in test_ai_names.py.

Policy (launch-schema.md "Build rules", Jason 19-20 Nov 2026):
- A seat is typed only when its printed name names an AI product, on a work published on or after that product's
  launch (less a 31-day grace for early access). The publication date decides; OpenAlex's own creation date stands in
  only when the publication date is missing or impossible (outside 1800 to next year), never to rescue an old work
  ingested late (outside reviews, 21 Nov 2026). Bylines are typed as published; we take no stance on whether an AI can be an
  author.
- Never a bare first name: "Claude", "Gemini", "Grok", "Kimi", "Llama", "Mistral", "Gemma", "Devin", "Sora", "Bard",
  "Codex" are names people carry, so they count only with a qualifier in the same seat: the maker (Anthropic, Google,
  xAI...), a model sub-name (Opus, Sonnet, Flash...), a version beside the name (Claude 3.5, Gemini 2.5), or an AI word
  (AI, assistant, chatbot, language model).
- Names no person carries (ChatGPT, GPT-4, DeepSeek, Midjourney...) count on their own.
- A maker's name alone (OpenAI, Anthropic, Google DeepMind) is an organization, not a software agent.
- A seat that also holds a person (an ORCID, a degree, "assisted by", two or more name words beside the product with
  no AI word) stays untyped: precision first.

Each product row: key, display name, maker, launch date (first public availability), kind, and its patterns:
`strong` (counts alone) or `name` (counts only with a qualifier). Dates are from the makers' own announcements.
"""

import html
import re
import unicodedata
from dataclasses import dataclass, field
from datetime import date, timedelta

RULE_VERSION = "ai-names-v4"
GRACE_DAYS = 31


@dataclass(frozen=True)
class Product:
    key: str
    display: str
    maker: str
    launch: str  # ISO date, first public availability
    kind: str
    strong: tuple = ()  # regexes that name the product on their own
    name: tuple = ()  # regexes that name it only with a qualifier
    sub: tuple = ()  # model sub-names that qualify the name ("opus")
    makers: tuple = ()  # maker words that qualify the name ("anthropic")


PRODUCTS: list[Product] = [
    Product("chatgpt", "ChatGPT", "OpenAI", "2022-11-30", "chat assistant",
            strong=(r"(?<![\w’'])chat\s?-?\s?gpt(?![a-z])",)),
    Product("gpt", "GPT (OpenAI models)", "OpenAI", "2019-02-14", "language model",
            strong=(r"(?<![\w’'])gpt\s?-?\s?(?:[2-9](?:\s?[.・]\s?\d{1,2})?|4o|4\.1|4\.5)(?![0-9])",
                    r"generative\s+pre-?\s?trained\s+transformer",
                    r"(?<![\w’'])o[134]-(?:mini|preview|pro|high)\b")),
    Product("claude", "Claude", "Anthropic", "2023-03-14", "chat assistant",
            name=(r"(?<![\w’'-])claude(?![\w’'])", r"(?<![\w’'-])клод(?![а-я])"),
            sub=(r"(?<![\w’'])(?:opus|sonnet|haiku|fable|instant)(?![\w’'])", r"claude\s?code", r"claude\.ai"),
            makers=(r"anthropic",)),
    Product("gemini", "Gemini", "Google", "2023-12-06", "chat assistant",
            name=(r"(?<![\w’'])gemini(?![\w’'])",),
            sub=(r"(?<![\w’'])(?:flash|pro|ultra|nano|advanced)(?![\w’'])", r"deep\s?research",
                 r"modello\s+linguistico"),
            makers=(r"google", r"deepmind")),
    Product("bard", "Bard", "Google", "2023-03-21", "chat assistant",
            name=(r"(?<![\w’'])bard(?![\w’'])",),
            makers=(r"google",)),
    Product("grok", "Grok", "xAI", "2023-11-04", "chat assistant",
            name=(r"(?<![\w’'])grok(?![\w’'])",),
            makers=(r"(?<![\w’'])x\s?ai(?![\w’'])", r"spacexai")),
    Product("deepseek", "DeepSeek", "DeepSeek", "2023-11-02", "chat assistant",
            strong=(r"(?<![\w’'])deep\s?-?seek(?![a-z])",)),
    Product("copilot", "Copilot", "Microsoft / GitHub", "2021-06-29", "chat and coding assistant",
            strong=(r"(?<![\w’'])co-?pilot(?![\w’'])", r"copilot\s?\.microsoft")),
    Product("bing_chat", "Bing Chat", "Microsoft", "2023-02-07", "chat assistant",
            strong=(r"bing\s+chat",)),
    Product("perplexity", "Perplexity", "Perplexity AI", "2022-12-07", "answer engine",
            strong=(r"(?<![\w’'])perplexity",)),
    Product("llama", "Llama", "Meta", "2023-02-24", "language model",
            name=(r"(?<![\w’'])llama(?![\w’'])",),
            makers=(r"(?<![\w’'])meta(?![\w’'])",)),
    Product("mistral", "Mistral models / Le Chat", "Mistral AI", "2023-09-27", "language model",
            name=(r"(?<![\w’'])mistral(?![\w’'])", r"mixtral"),
            sub=(r"le\s+chat", r"(?<![\w’'])(?:large|7b|8x7b|codestral)(?![\w’'])",)),
    Product("qwen", "Qwen", "Alibaba", "2023-04-11", "language model",
            name=(r"(?<![\w’'])qwen",), makers=(r"alibaba",)),
    Product("kimi", "Kimi", "Moonshot AI", "2023-10-09", "chat assistant",
            name=(r"(?<![\w’'])kimi(?![\w’'])",),
            sub=(r"(?<![\w’'])k\s?\d",),
            makers=(r"moonshot",)),
    Product("doubao", "Doubao", "ByteDance", "2023-08-17", "chat assistant",
            name=(r"(?<![\w’'])doubao(?![\w’'])",), strong=(r"豆包",), makers=(r"bytedance",)),
    Product("ernie_bot", "ERNIE Bot", "Baidu", "2023-03-16", "chat assistant",
            strong=(r"ernie\s?bot", r"文心一言")),
    Product("hunyuan", "Hunyuan", "Tencent", "2023-09-07", "language model",
            name=(r"(?<![\w’'])hunyuan(?![\w’'])",), strong=(r"混元",), makers=(r"tencent",)),
    Product("chatglm", "ChatGLM / GLM", "Zhipu AI", "2023-03-14", "language model",
            strong=(r"(?<![\w’'])chatglm", r"(?<![\w’'])glm-\d")),
    Product("gemma", "Gemma", "Google", "2024-02-21", "language model",
            name=(r"(?<![\w’'])gemma(?![\w’'])",),
            makers=(r"google",)),
    Product("phi", "Phi", "Microsoft", "2023-06-20", "language model",
            strong=(r"(?<![\w’'])phi-[1-4](?:\.\d)?(?![0-9])",)),
    Product("nemotron", "Nemotron", "NVIDIA", "2024-06-14", "language model",
            strong=(r"(?<![\w’'])nemotron",)),
    Product("huggingchat", "HuggingChat", "Hugging Face", "2023-04-25", "chat assistant",
            strong=(r"(?<![\w’'])huggingchat",)),
    Product("notebooklm", "NotebookLM", "Google", "2023-07-12", "research assistant",
            strong=(r"(?<![\w’'])notebook\s?lm",)),
    Product("codex", "Codex", "OpenAI", "2021-08-10", "coding agent",
            name=(r"(?<![\w’'])codex(?![\w’'])",),
            sub=(r"codex\s?cli",),
            makers=(r"open\s?ai",)),
    Product("devin", "Devin", "Cognition", "2024-03-12", "coding agent",
            name=(r"(?<![\w’'])devin(?![\w’'])",),
            makers=(r"cognition",)),
    Product("manus", "Manus", "Butterfly Effect", "2025-03-06", "agent",
            name=(r"(?<![\w’'])manus(?![\w’'])",)),
    Product("sora", "Sora", "OpenAI", "2024-02-15", "video model",
            name=(r"(?<![\w’'])sora(?![\w’'])",),
            makers=(r"open\s?ai",)),
    Product("dalle", "DALL-E", "OpenAI", "2021-01-05", "image model",
            strong=(r"(?<![\w’'])dall[-·]e(?![\w’'])",)),
    Product("midjourney", "Midjourney", "Midjourney", "2022-07-12", "image model",
            strong=(r"(?<![\w’'])midjourney",)),
    Product("stable_diffusion", "Stable Diffusion", "Stability AI", "2022-08-22", "image model",
            strong=(r"stable\s+diffusion",)),
    Product("ai_scientist", "The AI Scientist", "Sakana AI", "2024-08-12", "research agent",
            strong=(r"the\s+ai\s+scientist",)),
    Product("ai_coscientist", "AI co-scientist", "Google", "2025-02-19", "research agent",
            strong=(r"ai\s+co-?\s?scientist",)),
    Product("zochi", "Zochi", "Intology", "2025-03-01", "research agent",
            name=(r"(?<![\w’'])zochi(?![\w’'])",), makers=(r"intology",)),
    Product("minimax", "MiniMax Agent", "MiniMax", "2023-06-01", "agent",
            name=(r"(?<![\w’'])minimax(?![\w’'])",), sub=(r"minimax\s+agent", r"hailuo")),
    Product("alphaxiv_agent", "alphaXiv agent", "alphaXiv", "2025-08-01", "research agent",
            strong=(r"^agent\s+[a-z]+-[0-9a-f]{8}$",)),
]

# Maker names: alone they are organizations (the company), and they qualify a product name.
MAKER_ORGS = [
    (r"open\s?ai(?:\s?,?\s?(?:inc\.?|llc|l\.?p\.?|\(united states\)))?", "OpenAI"),
    (r"anthropic(?:\s?,?\s?(?:pbc|inc\.?|\(united states\)))?", "Anthropic"),
    (r"(?:google\s+)?deepmind(?:\s+technologies)?", "Google DeepMind"),
    (r"x\.?ai(?:\s+corp\.?)?", "xAI"),
    (r"deepseek[-\s]?ai|deepseek\s*\(china\)|deepseek\s+(?:ai\s+)?lab", "DeepSeek"),
    (r"mistral\s+ai", "Mistral AI"),
    (r"moonshot\s+ai", "Moonshot AI"),
    (r"zhipu(?:\s+ai)?", "Zhipu AI"),
    (r"qwen\s+team", "Qwen Team (Alibaba)"),
    (r"midjourney,?\s+inc\.?", "Midjourney"),
    (r"sakana\s+ai", "Sakana AI"),
    (r"stability\s+ai", "Stability AI"),
    (r"perplexity\s+ai,?\s+inc\.?", "Perplexity AI"),
]
MAKER_ORG_RE = re.compile(r"^(?:" + "|".join(p for p, _ in MAKER_ORGS) + r")$")

# Words that say the seat is an AI (qualify an ambiguous product name).
AI_WORDS = re.compile(
    r"\[bot\]|(?<![\w’'])(?:ai|a\.i\.|llm|chatbot|assistant|assistente|assistante|asistente|"
    r"language\s+model|ai\s+model|modello\s+linguistico|intelligence\s+artificielle|"
    r"künstliche\s+intelligenz|inteligencia\s+artificial|artificial\s+intelligence)(?![\w’'])")
# "IA" (fr/es/it/pt) and "KI" (de) count only in capitals: in lower case they are pieces of names (Mékiès, Andréia).
AI_CAPS = re.compile(r"(?<![\w’'])(?:IA|KI)(?![\w’'])")
# Web addresses other than the products' own: the entry is a website (bar v1: "ChatGPT Deutsch, ChatGPTDeutsch.info").
PRODUCT_DOMAINS = re.compile(r"(?:claude\.ai|chatgpt\.com|chat\.openai\.com|gemini\.google\.com|perplexity\.ai|"
                             r"copilot\.microsoft\.com|poe\.com|grok\.com|chat\.deepseek\.com|qwen\.ai|kimi\.ai)")
DOMAIN = re.compile(r"[a-z0-9-]+\.(?:com|info|net|org|io|ai|de|fr|ru|cn|co|app|xyz|me)(?![\w’'])")
# The team or company behind a product ("Qwen Team (Alibaba Cloud)") is an organization, not the product.
ORG_AFTER_PRODUCT = re.compile(r"(?<![\w’'])(?:team|lab|labs|laboratory|group|inc\.?|ltd\.?|llc|corp\.?|corporation|"
                               r"company|research\s+(?:team|group|lab))(?![\w’'])|(?<![\w’'])research\s*$")  # "OpenAI Codex Research"
# Another name joined to the product in one entry ("Pion + Claude Sonnet 4.6"): two contributors, untyped (R3).
JOINED = re.compile(r"\s[+&]\s?|\s?[+&]\s|(?<![\w’'])(?:and|und|et|y|e|with|mit|avec|con)(?![\w’'])")
VERSION_AFTER = r"\s?-?\s?v?(\d{1,2}(?:\s?[.・]\s?\d{1,2})?)(?![0-9])"

# Signs that the seat holds a person or is not a byline at all.
HUMAN_GUARD = re.compile(
    r"\d{4}-\d{4}-\d{4}-\d{3}[\dx]|@[a-z0-9-]+\.|(?<![\w’'])(?:ph\.?\s?d|m\.?d\.|msc|mba|esq|llb|jd|prof\.?|dr\.?)(?![\w’'])|"
    r"(?<![\w’'])(?:using|utili[sz]ing|leveraging|adapting|assisted\s+by|with\s+the\s+help\s+of|aided\s+by|"
    r"supported\s+by|in\s+collaboration\s+with|generated\s+with|created\s+with|written\s+with)(?![\w’'])")
INSTITUTION_WORDS = re.compile(
    r"(?<![\w’'])(?:universit[a-zéà]*|university|hospital|hôpital|institut[a-z]*|college|collège|école|lycée|"
    r"school|faculty|department|laboratory|association|society|foundation|ministry|council|committee)(?![\w’'])")
# A seat that names no product but calls itself an AI ("Anthropic AI Assistant", "Haru, language model").
GENERIC_AI = re.compile(
    r"(?<![\w’'])(?:(?:ai|a\.i\.|ia|ki)[\s-]+(?:[a-z]+\s+)?(?:assistant|assistante|asistente|assistente|assistent|model|modèle|"
    r"collaborator|co-?author|agent|chatbot)|(?:large\s+)?language\s+model|chatbot|(?:assistant|modèle|model)\s+(?:ia|ai))(?![\w’'])")
GENERIC_LAUNCH = "2022-11-30"  # ChatGPT: the first chat assistant people credited on bylines
MAKER_WORDS = re.compile(r"(?<![\w’'])(?:open\s?ai|anthropic|google|deepmind|microsoft|x\.?ai|meta|deepseek|alibaba|"
                         r"baidu|bytedance|tencent|mistral|moonshot)(?![\w’'])")
NAME_TOKEN = re.compile(r"^[A-ZÀ-ÖØ-Þ][a-zà-öø-ÿ'’-]{2,}$")
ROLE_WORDS = set("""ai a.i. ia ki llm assistant assistente assistante asistente chatbot bot model language models
large co-author coauthor co author collaborator collaborative contributor computational research system systems
agent agente role persona thinking reasoning extended fast mini preview pro flash ultra nano advanced instant
code chat the and of by et y e und with for als as panel team collective kollektiv simulation generative
artificial intelligence pretrained pre-trained transformer developed via google anthropic openai open microsoft
github meta xai deepmind alibaba cloud moonshot cognition partner cognitive sonnet opus haiku fable claude gemini
grok chatgpt gpt deepseek copilot perplexity llama mistral qwen kimi gemma codex devin sora bard manus
modello linguistico di""".split())


def norm(s: str) -> str:
    s = unicodedata.normalize("NFKC", html.unescape(s or ""))  # "M&eacute;ki&egrave;s" is Mékiès, not "ki" (bar v1)
    return re.sub(r"\s+", " ", s).strip()


def _compile(p: Product):
    return (
        [re.compile(x) for x in p.strong],
        [re.compile(x) for x in p.name],
        [re.compile(x) for x in p.sub],
        [re.compile(x) for x in p.makers],
    )


_COMPILED = [(p, *_compile(p)) for p in PRODUCTS]


@dataclass
class Verdict:
    type: str | None  # 'software-agent' | 'organization' | None
    rule: str  # why
    products: list = field(default_factory=list)  # product keys named
    launch: str | None = None  # earliest launch among products named (the date guard uses the LATEST of them? no: earliest)
    maker: str | None = None


def _leftover_name_tokens(raw: str, low: str) -> list[str]:
    """Capitalized name-like words left after removing product, maker, version and role words."""
    toks = re.findall(r"[^\s,;:()\[\]/•·—–|\"“”+&]+", raw)  # "+" and "&" join two names (bar v3: "Pion+ Claude")
    out = []
    for t in toks:
        tl = t.lower().strip(".-'’")
        if not tl or tl in ROLE_WORDS or re.fullmatch(r"[\d.・]+[a-z]?", tl) or len(tl) <= 2:
            continue
        if any(rx.search(tl) for p, st, nm, sb, mk in _COMPILED for rx in st + nm):
            continue
        if NAME_TOKEN.match(t.strip(".")):
            out.append(t)
    return out


def type_name(raw: str) -> Verdict:
    """Type one printed byline name (no date guard; see type_seat)."""
    s = norm(raw)
    low = s.lower()
    if not low:
        return Verdict(None, "empty")
    if MAKER_ORG_RE.match(low.strip(" .")):
        # A maker's name alone: the company, or its chatbot credited by the company's name (bar v1: judges read 6 of
        # 80 such entries on self-deposits as the chatbot). Untyped by this list; the organization lane decides.
        return Verdict(None, "ai_maker_alone", maker=low)
    hits = []  # (product, how)
    for p, st, nm, sb, mk in _COMPILED:
        if any(rx.search(low) for rx in st):
            hits.append((p, "strong"))
            continue
        for rx in nm:
            m = rx.search(low)
            if not m:
                continue
            after = low[m.end():]
            if re.match(VERSION_AFTER, after):
                v = re.match(VERSION_AFTER, after).group(1)
                try:
                    if 1 <= float(re.sub(r"\s|・", ".", v).replace("..", ".")) <= 10:
                        hits.append((p, "name+version"))
                        break
                except ValueError:
                    pass
            if any(re.match(r"[\s\-.(/]*(?:" + x.pattern + ")", after) or x.match(low, m.start()) or (x.pattern.startswith(p.key) and x.search(low))
                   or re.search("(?:" + x.pattern + ")" + VERSION_AFTER, low) for x in sb):
                hits.append((p, "name+subname"))
                break
            if any(x.search(low) for x in mk):
                hits.append((p, "name+maker"))
                break
            if (AI_WORDS.search(low) or AI_CAPS.search(s)) and not re.fullmatch(r"a\.?\s?i\.?\s+\S+", low):
                hits.append((p, "name+ai_word"))
                break
    if not hits:
        if GENERIC_AI.search(low):
            return _generic(s, low)
        return Verdict(None, "no_product")
    # A seat naming two or more products (even ambiguous ones) is an AI panel when one of them qualified.
    keys = sorted({p.key for p, _ in hits})
    launch = min(p.launch for p, _ in hits)
    if HUMAN_GUARD.search(low):
        return Verdict(None, "human_guard", keys, launch)
    if INSTITUTION_WORDS.search(low) and not AI_WORDS.search(low):
        return Verdict(None, "institution_words", keys, launch)
    if len(s.split()) > 12 and len(keys) < 2:
        return Verdict(None, "fragment", keys, launch)
    if ORG_AFTER_PRODUCT.search(low) and not GENERIC_AI.search(low):
        return Verdict(None, "org_words", keys, launch)
    if DOMAIN.search(PRODUCT_DOMAINS.sub(" ", low)):
        return Verdict(None, "website", keys, launch)
    left = _leftover_name_tokens(s, low)
    if left and JOINED.search(low):
        return Verdict(None, "joined_with_other", keys, launch)
    if len(left) >= 2 and not GENERIC_AI.search(low):
        return Verdict(None, "person_beside_product", keys, launch)
    how = "+".join(sorted({h for _, h in hits}))
    rule = ("persona+" if left else "") + how
    return Verdict("software-agent", rule, keys, launch)


def _generic(s: str, low: str) -> Verdict:
    """No product named, but the seat calls itself an AI: typed only when every word is an AI or role word, besides at
    most one persona name ("AI Assistant, Max"), and no person signs."""
    if HUMAN_GUARD.search(low) or INSTITUTION_WORDS.search(low):
        return Verdict(None, "generic_guard")
    words = [w.lower().strip(".-'’") for w in re.findall(r"[^\s,;:()\[\]/•·—–|\"“”]+", s)]
    other = [w for w in words if w and w not in ROLE_WORDS and not re.fullmatch(r"[\d.]+", w)]
    if len(other) > 1:
        return Verdict(None, "generic_other_words")
    rule = "generic_ai+maker" if MAKER_WORDS.search(low) else "generic_ai"
    if other:
        rule = "persona+" + rule
    return Verdict("software-agent", rule, ["generic"], GENERIC_LAUNCH)


def type_seat(raw: str, pub_date: date | None, created_date: date | None) -> Verdict:
    """Type a seat: the name rule, then the launch-date guard on the later of publication and OpenAlex creation date."""
    v = type_name(raw)
    if v.type != "software-agent":
        return v
    valid = pub_date is not None and date(1800, 1, 1) <= pub_date <= date(date.today().year + 1, 12, 31)
    eff = pub_date if valid else (created_date or date(1, 1, 1))
    launch = date.fromisoformat(v.launch) - timedelta(days=GRACE_DAYS)
    if eff < launch:
        return Verdict(None, "before_launch", v.products, v.launch)
    return v


def product_table() -> list[dict]:
    return [dict(key=p.key, display=p.display, maker=p.maker, launch=p.launch, kind=p.kind,
                 strong=list(p.strong), name_only_with_qualifier=list(p.name), subnames=list(p.sub),
                 maker_words=list(p.makers)) for p in PRODUCTS]


# A cheap SQL prefilter (Java regex, run on LOWER(raw_author_name)) that every name type_name can type must match.
# Looser than the rules on purpose: it only keeps the UDF off the ~1.5B seats that can't match. test_ai_names checks
# every typed test string passes it.
SQL_PREFILTER = (
    r"chat\s?-?\s?gpt|gpt|generative\s+pre-?\s?trained|claude|клод|gemini|bard|grok|deep\s?-?seek|co-?pilot|"
    r"bing\s+chat|perplexity|llama|mistral|mixtral|qwen|kimi|doubao|豆包|ernie\s?bot|文心一言|hunyuan|混元|chatglm|glm-\d|"
    r"gemma|phi-\d|nemotron|huggingchat|notebook\s?lm|codex|devin|manus|sora|dall[-·]e|midjourney|stable\s+diffusion|"
    r"ai\s+scientist|ai\s+co-?\s?scientist|zochi|minimax|^agent\s|o[134]-|open\s?ai|anthropic|deepmind|x\.?ai|zhipu|"
    r"sakana|stability\s+ai|moonshot|qwen\s+team|chatbot|language\s+model|"
    r"(ai|a\.i\.|ia|ki)[\s-]+([a-z]+\s+)?(assistant|assistante|asistente|assistente|assistent|model|modèle|"
    r"collaborator|co-?author|agent|chatbot)|(assistant|modèle|model)\s+(ia|ai)"
)

v = type_name(s)
return {"type": v.type, "rule": v.rule, "products": v.products, "launch": v.launch}
$$

In [ ]:
-- oxjob #2437: the typed-seat table (same definition as CreateWorkAuthorships' cell).
CREATE TABLE IF NOT EXISTS openalex.works.work_author_types (
  work_id         BIGINT,
  author_sequence BIGINT,
  raw_author_name STRING,
  type            STRING,
  rule            STRING,
  rule_version    STRING,
  evidence        STRING
)
COMMENT 'oxjob #2437: author type per byline seat (work_id, author_sequence as in work_authors); one row per typed seat, no row = not stated. Read by CreateWorkAuthorships.'
CLUSTER BY (work_id)

In [ ]:
-- oxjob #2437: tonight's candidate seats: work_authors rows whose lowered name passes the prefilter (ai_names.SQL_PREFILTER,
-- looser than the rule on purpose), with the name the base byline prints at that seat (the name typed, and the one
-- CreateWorkAuthorships checks), the work's title and its dates. created_date: the earliest date OpenAlex has served
-- for the work (openalex_works_created_date, #1873), else tonight's derived one.
CREATE OR REPLACE TABLE openalex.authors.ai_byline_candidate_seats
COMMENT 'oxjob #2437: byline seats whose name passes the AI-name prefilter (candidates, not typed). Rebuilt by TypeAuthorSeats each run.'
AS
WITH p AS (
  SELECT DISTINCT work_id, author_sequence
  FROM openalex.works.work_authors
  WHERE LOWER(raw_author_name) RLIKE 'chat\\s?-?\\s?gpt|gpt|generative\\s+pre-?\\s?trained|claude|клод|gemini|bard|grok|deep\\s?-?seek|co-?pilot|bing\\s+chat|perplexity|llama|mistral|mixtral|qwen|kimi|doubao|豆包|ernie\\s?bot|文心一言|hunyuan|混元|chatglm|glm-\\d|gemma|phi-\\d|nemotron|huggingchat|notebook\\s?lm|codex|devin|manus|sora|dall[-·]e|midjourney|stable\\s+diffusion|ai\\s+scientist|ai\\s+co-?\\s?scientist|zochi|minimax|^agent\\s|o[134]-|open\\s?ai|anthropic|deepmind|x\\.?ai|zhipu|sakana|stability\\s+ai|moonshot|qwen\\s+team|chatbot|language\\s+model|(ai|a\\.i\\.|ia|ki)[\\s-]+([a-z]+\\s+)?(assistant|assistante|asistente|assistente|assistent|model|modèle|collaborator|co-?author|agent|chatbot)|(assistant|modèle|model)\\s+(ia|ai)'
)
SELECT p.work_id, p.author_sequence, GET(b.authorships, CAST(p.author_sequence AS INT)).raw_author_name AS raw_author_name,
       b.title, b.publication_date, LEAST(cd.created_date, b.created_date) AS created_date
FROM p
JOIN openalex.works.openalex_works_base b ON b.id = p.work_id
LEFT JOIN openalex.works.openalex_works_created_date cd ON cd.id = p.work_id
WHERE LOWER(GET(b.authorships, CAST(p.author_sequence AS INT)).raw_author_name) RLIKE 'chat\\s?-?\\s?gpt|gpt|generative\\s+pre-?\\s?trained|claude|клод|gemini|bard|grok|deep\\s?-?seek|co-?pilot|bing\\s+chat|perplexity|llama|mistral|mixtral|qwen|kimi|doubao|豆包|ernie\\s?bot|文心一言|hunyuan|混元|chatglm|glm-\\d|gemma|phi-\\d|nemotron|huggingchat|notebook\\s?lm|codex|devin|manus|sora|dall[-·]e|midjourney|stable\\s+diffusion|ai\\s+scientist|ai\\s+co-?\\s?scientist|zochi|minimax|^agent\\s|o[134]-|open\\s?ai|anthropic|deepmind|x\\.?ai|zhipu|sakana|stability\\s+ai|moonshot|qwen\\s+team|chatbot|language\\s+model|(ai|a\\.i\\.|ia|ki)[\\s-]+([a-z]+\\s+)?(assistant|assistante|asistente|assistente|assistent|model|modèle|collaborator|co-?author|agent|chatbot)|(assistant|modèle|model)\\s+(ia|ai)'

In [ ]:
-- oxjob #2437: one verdict per distinct candidate name. The only statement that calls the UDF; the next cell reads the
-- table, so the warehouse never calls it twice for a filter on its result.
-- shortcut: every distinct name is judged again each night (239,067 names, ~4 min on an X-Large, 21 Nov 2026); keep
-- last night's verdicts per (name, rule version) and judge only new names if this grows past ~15 min.
CREATE OR REPLACE TABLE openalex.authors.ai_byline_name_verdicts
COMMENT 'oxjob #2437: ai-names-v4 verdict per distinct candidate byline name. Rebuilt by TypeAuthorSeats each run.'
AS
SELECT raw_author_name, openalex.authors.type_ai_byline_name(raw_author_name) AS v
FROM (SELECT DISTINCT raw_author_name FROM openalex.authors.ai_byline_candidate_seats WHERE raw_author_name IS NOT NULL)

In [ ]:
-- oxjob #2437: tonight's proposed seats. The title guard: a seat whose name is the work's own title ("ChatGPT" as the
-- author of a work titled "ChatGPT") is a mis-parsed byline, left untyped. The launch guard: a work dated more than
-- 31 days before the product it names launched is a namesake; the publication date decides (the first-seen date
-- only when it is missing or outside 1800 to next year). Proposed, not typed: the confirmer keeps what the model confirms.
CREATE OR REPLACE TABLE openalex.authors.ai_byline_rule_seats
COMMENT 'oxjob #2437: seats the AI-name rules propose as software agents tonight (before the model confirmer). Rebuilt by TypeAuthorSeats each run.'
AS
SELECT c.work_id, c.author_sequence, c.raw_author_name, n.v.rule AS rule, 'ai-names-v4' AS rule_version,
       TO_JSON(NAMED_STRUCT('products', n.v.products, 'launch', n.v.launch, 'work_date', CASE WHEN c.publication_date BETWEEN DATE'1800-01-01' AND DATE_ADD(CURRENT_DATE(), 366) THEN c.publication_date ELSE COALESCE(c.created_date, DATE'0001-01-01') END)) AS evidence
FROM openalex.authors.ai_byline_candidate_seats c
JOIN openalex.authors.ai_byline_name_verdicts n ON n.raw_author_name = c.raw_author_name
WHERE n.v.type = 'software-agent'
  AND NOT (TRIM(REGEXP_REPLACE(LOWER(c.raw_author_name), '[^\\p{L}\\p{N}]+', ' ')) <=> TRIM(REGEXP_REPLACE(LOWER(c.title), '[^\\p{L}\\p{N}]+', ' ')))
  AND COALESCE(CASE WHEN c.publication_date BETWEEN DATE'1800-01-01' AND DATE_ADD(CURRENT_DATE(), 366) THEN c.publication_date ELSE COALESCE(c.created_date, DATE'0001-01-01') END >= DATE_SUB(TO_DATE(n.v.launch), 31), TRUE)

In [ ]:
-- oxjob #2437: the card fields for every proposed seat, as the bar's cards (oxjobs #2437 work/build/bar/cards_all_agent.sql),
-- read from tonight's base: the byline's first 40 names, title, venue and publisher of the primary location, year, work
-- type, DOI, and the affiliation text printed for this seat. The confirmer renders them (delimited, capped) and hashes
-- the card with the model and prompt version; it asks only about cards with no answer on file.
CREATE OR REPLACE TABLE openalex.authors.ai_byline_cards
COMMENT 'oxjob #2437: proposed software-agent seats with their card fields, for the model confirmer. Rebuilt by TypeAuthorSeats each run.'
AS
SELECT r.work_id, r.author_sequence, r.raw_author_name, r.rule, r.rule_version, r.evidence,
       b.title, b.publication_year, b.type AS work_type, b.doi,
       b.primary_location.source.display_name AS source_name,
       b.primary_location.source.host_organization_name AS publisher,
       TRANSFORM(SLICE(b.authorships, 1, 40), x -> x.raw_author_name) AS byline,
       SIZE(b.authorships) AS n_authors,
       GET(b.authorships, CAST(r.author_sequence AS INT)).raw_affiliation_strings AS seat_affiliations
FROM openalex.authors.ai_byline_rule_seats r
JOIN openalex.works.openalex_works_base b ON b.id = r.work_id

In [ ]:
-- oxjob #2437: tonight's proposals (display only)
SELECT rule, COUNT(*) AS seats, COUNT(DISTINCT work_id) AS works FROM openalex.authors.ai_byline_cards GROUP BY rule ORDER BY seats DESC